# OpenPyTEA — Tutorial 2: Creating a Plant

A `Plant` in OpenPyTEA aggregates all equipment, economic parameters, operating costs, and revenue assumptions into a single model. Once configured, it can compute:

- **Capital expenditure (CAPEX)** — fixed capital investment and its breakdown
- **Operating expenditure (OPEX)** — both fixed (labour, maintenance) and variable (feedstocks, utilities)
- **Revenue** — annual revenue from plant products
- **Cash flow** — year-by-year cash flow with CAPEX profile, depreciation, and taxes
- **Financial metrics** — levelized cost of production (LCOP), net present value (NPV), internal rate of return (IRR), return on investment (ROI), and payback time

In this tutorial, we will build an **arbitrary plant configuration** to demonstrate how to set up and run a full techno-economic analysis using OpenPyTEA.

---

In [1]:
from openpytea import Equipment, Plant

## Step 1 — Define the Equipment

We create the same four pieces of equipment introduced in Tutorial 1.

In [2]:
compressor = Equipment(
    name="COMP-1",
    param=100,
    process_type="Fluids",
    category="Compressors, fans, & blowers",
    type="Compressor, centrifugal",
    material="Carbon steel",
)

heat_exchanger = Equipment(
    name="HX-1",
    param=35,
    process_type="Fluids",
    category="Heat Exchangers",
    type="U-tube shell & tube",
    material="304 stainless steel",
)

reactor = Equipment(
    name="REACT-1",
    param=0,
    process_type="Fluids",
    category="Reactors",
    purchased_cost=1_500_000,
    cost_year=2019,
)

pump = Equipment(
    name="PUMP-1",
    param=10,
    process_type="Fluids",
    category="Pumps",
    type="Centrifugal",
    material="Carbon steel",
)

equipment_list = [compressor, heat_exchanger, reactor, pump]

## Step 2 — Write the Plant Configuration

The `Plant` constructor takes a single Python dictionary:

In [3]:
config = {
    # --- Plant identity ---
    "plant_name": "Methanol Synthesis",     # A label for the Plant
    "process_type": "Fluids",               # The type of process ("Fluids", "Solids", or "Mixed")

    # --- Plant location ---
    "country": "United States",             # country where the plant is located                
    "region": "Gulf Coast",                 # region within the country  

    # --- Equipment ---
    "equipment": equipment_list,            # list of Equipment objects in the plant

    # --- Economic parameters ---
    "interest_rate":     0.08,              # 8 % discount rate
    "project_lifetime":  20,                # years
    "plant_utilization": 0.90,              # 90 % capacity factor
    "tax_rate":          0.25,              # 25 % corporate tax

    # --- Labour ---
    "operator_hourly_rate": {
        "rate": 38.0,     # USD/hr (mean)
    },

    # --- Products ---
    # production: kg/day    price: unit price
    "plant_products": {
        "methanol": {
            "production": 45_000,  # kg/day
            "price":      1.0,     # USD/kg
        },
        "carbon_dioxide": {
            "production": 10_000,  # kg/day
            "price":      0.6,     # USD/kg
        }
    },

    # --- Variable operating costs ---
    # consumption: daily amount    price: unit cost
    "variable_opex_inputs": {
        "natural_gas": {
            "consumption": 35_000,  # kg/day
            "price":       0.4,     # USD/kg
        },
        "process_water": {
            "consumption": 20_000,  # kg/day
            "price":       0.002,   # USD/kg
        },
        "cooling_water": {
            "consumption": 5_000,  # kg/day
            "price":       0.0005, # USD/kg
        },
        "electricity": {
            "consumption": 10_000, # kWh/day
            "price":       0.08,   # USD/kWh
        }
    },

    # --- Depreciation ---
    "depreciation": {
        "method": "straight_line",
        "life": 12,
        "salvage_fraction": 0.05,
        "service_start_year": 2
    }
}

## Step 3 — Instantiate the Plant

In [4]:
plant = Plant(config)

## Step 4 — Run the Calculations

Call individual methods to compute specific values, or use `calculate_all()` to run everything at once. Individual methods can then return the cached results.

In [5]:
# Capital cost breakdown (set print_results=True for a detailed table)
plant.calculate_fixed_capital(print_results=True)

Capital cost estimation
ISBL: 10,695,255.77 USD
OSBL: 3,208,576.73 USD
Design and engineering: 4,171,149.75 USD
Contingency: 1,390,383.25 USD
Fixed capital investment: 19,465,365.50 USD


In [6]:
# ISBL breakdown (set print_results=True for a detailed table)
plant.calculate_isbl(print_results=True)

ISBL cost estimation
  - COMP-1: 4,168,963.90 USD
  - HX-1: 173,003.42 USD
  - REACT-1: 6,320,987.65 USD
  - PUMP-1: 32,300.79 USD
Total ISBL: 10,695,255.77 USD


In [7]:
# Annual operating costs
plant.calculate_variable_opex(print_results=True)

Variable production costs estimation
  - Natural gas: 4,599,000.00 USD per year
  - Process water: 13,140.00 USD per year
  - Cooling water: 821.25 USD per year
  - Electricity: 262,800.00 USD per year
Total Variable OPEX: 4,875,761.25USD per year


In [8]:
plant.calculate_fixed_opex(print_results=True)

Fixed production costs estimation
Operating labor costs: 893,760.00 USD per year
Supervision costs: 223,440.00 USD per year
Direct salary overhead: 558,600.00 USD per year
Laboratory charges: 89,376.00 USD per year
Maintenance costs: 534,762.79 USD per year
Taxes and insurance costs: 160,428.84 USD per year
Rent of land costs: 208,557.49 USD per year
Environmental charges: 139,038.32 USD per year
Operating supplies: 96,257.30 USD per year
General plant overhead: 1,089,270.00 USD per year
Interest on working capital: 233,584.39 USD per year
Patents and royalties: 195,759.92 USD per year
Distribution and selling costs: 195,759.92 USD per year
R&D costs: 293,639.88 USD per year
Fixed OPEX: 4,912,234.85 USD per year


In [9]:
# Annual revenue
plant.calculate_revenue(print_results=True)

Revenue estimation
  - Methanol: 14,782,500.00 USD per year
  - Carbon dioxide: 1,971,000.00 USD per year
Total Revenue: 16,753,500.00 USD per year


In [10]:
# Key financial metrics
plant.calculate_levelized_cost(print_results=True)
plant.calculate_roi(print_results=True)
plant.calculate_irr(print_results=True)
plant.calculate_payback_time(print_results=True)
plant.calculate_npv(print_results=True)

Levelized cost: 0.807 USD/unit
Return of investment: 18.54%
Internal Rate of Return: 12.94%
Payback time: 3.85 years
Year | Present Value [USD] | Cumulative NPV [USD]
-------------------------------------------
   1 |   -9,955,411.58 |   -9,955,411.58
   2 |  -14,224,497.73 |  -24,179,909.31
   3 |   -3,990,976.29 |  -28,170,885.60
   4 |    3,373,754.79 |  -24,797,130.81
   5 |    4,221,839.21 |  -20,575,291.60
   6 |    3,534,860.87 |  -17,040,430.73
   7 |    3,273,019.32 |  -13,767,411.40
   8 |    3,030,573.45 |  -10,736,837.96
   9 |    2,806,086.53 |   -7,930,751.43
  10 |    2,598,228.26 |   -5,332,523.17
  11 |    2,405,766.91 |   -2,926,756.25
  12 |    2,227,561.96 |     -699,194.30
  13 |    2,062,557.37 |    1,363,363.07
  14 |    1,909,775.34 |    3,273,138.40
  15 |    1,768,310.50 |    5,041,448.90
  16 |    1,524,873.14 |    6,566,322.05
  17 |    1,411,919.58 |    7,978,241.62
  18 |    1,307,332.94 |    9,285,574.56
  19 |    1,210,493.46 |   10,496,068.03
  20 |    

12243334.19819291

In [11]:
# Run all calculations in one call
plant.calculate_all(print_results=True)

Capital cost estimation
ISBL: 10,695,255.77 USD
OSBL: 3,208,576.73 USD
Design and engineering: 4,171,149.75 USD
Contingency: 1,390,383.25 USD
Fixed capital investment: 19,465,365.50 USD
Variable production costs estimation
  - Natural gas: 4,599,000.00 USD per year
  - Process water: 13,140.00 USD per year
  - Cooling water: 821.25 USD per year
  - Electricity: 262,800.00 USD per year
Total Variable OPEX: 4,875,761.25USD per year
Fixed production costs estimation
Operating labor costs: 893,760.00 USD per year
Supervision costs: 223,440.00 USD per year
Direct salary overhead: 558,600.00 USD per year
Laboratory charges: 89,376.00 USD per year
Maintenance costs: 534,762.79 USD per year
Taxes and insurance costs: 160,428.84 USD per year
Rent of land costs: 208,557.49 USD per year
Environmental charges: 139,038.32 USD per year
Operating supplies: 96,257.30 USD per year
General plant overhead: 1,089,270.00 USD per year
Interest on working capital: 233,584.39 USD per year
Patents and royaltie

## Step 5 — Update the Configuration

Use `update_configuration()` to change one or more parameters without rebuilding the entire plant object. Calculations are **not** automatically re-run — call `calculate_all()` (or the relevant individual method) after each update.

In [12]:
# Scenario: what if the discount rate rises to 12 %?
print(f"NPV at  8 % discount rate: ${plant.npv:>14,.0f}")

plant.update_configuration({"interest_rate": 0.12})
plant.calculate_all()
print(f"NPV at 12 % discount rate: ${plant.calculate_npv():>14,.0f}")

NPV at  8 % discount rate: $    12,243,334
NPV at 12 % discount rate: $       960,442


In [13]:
# Restore the original interest rate
plant.update_configuration({"interest_rate": 0.08})
plant.calculate_all()

## Step 6 — Override Capital Cost Factors

By default, OpenPyTEA derives OSBL, design & engineering, and contingency from the `process_type`. You can override any of these individually — as well as the location factor — via `update_configuration()`. This is useful when you have project-specific estimates or want to test different assumptions.

There are two override mechanisms:

**`fixed_capital_factors`** — override the multipliers (fractions) used to derive each component:

| Key | Default (Fluids) | Description |
|---|---|---|
| `"osbl"` | 0.30 × ISBL | Off-sites & battery limits |
| `"de"` | 0.30 × (ISBL + OSBL) | Design & engineering |
| `"contingency"` | 0.10 × (ISBL + OSBL) | Contingency allowance |

**`fixed_capital_components`** — override the computed cost value directly (takes precedence over `fixed_capital_factors` for the same component)

The `loc_factor` key overrides the country/region lookup and can be set independently.

In [14]:
# Default capital cost
print(f"Default FCI : ${plant.fixed_capital:>14,.0f}\n")

# Override using fixed_capital_factors: higher contingency and a custom location factor
plant.update_configuration({
    "fixed_capital_factors": {
        "contingency": 0.15,   # 15 % instead of 10 %
    },
    "fixed_capital_components": {
        "osbl": 3_700_000,     # fixed $ value
    },
    "loc_factor": 1.10,        # custom location factor
})

plant.calculate_fixed_capital(print_results=True)

Default FCI : $    19,465,365

Capital cost estimation
ISBL: 11,764,781.35 USD
OSBL: 3,700,000.00 USD
Design and engineering: 4,639,434.40 USD
Contingency: 2,319,717.20 USD
Fixed capital investment: 22,423,932.95 USD


In [15]:
# Restore defaults
plant.update_configuration({
    "contingency_factor": None,
    "loc_factor": None,
})
plant.calculate_all()

## Step 7 — Override Fixed OPEX Factors and Components

Fixed OPEX is built from a set of multipliers applied to labour, ISBL, and production costs. There are two ways to customise it:

- **`fixed_opex_factors`** — override one or more of the default multipliers. Omitted keys fall back to defaults.
- **`fixed_opex_components`** — set the absolute cost of a component directly (e.g. a known maintenance budget). Takes precedence over `fixed_opex_factors` for the same component.

Both are passed via `update_configuration()` and merged with the existing values — you only need to specify the keys you want to change.

### Labour cost estimation

Operating labour costs are auto-calculated from the number of process steps, the shift schedule, and the operator hourly rate. You can customise any part of this:

| Config key | Default | Description |
|---|---|---|
| `operator_hourly_rate` | `{"rate": 38.11}` | Operator wage, USD/hr |
| `working_weeks_per_year` | 49 | Working weeks per year |
| `working_shifts_per_week` | 5 | Shifts per working week |
| `operating_shifts_per_day` | 3 | Operating shifts per day |
| `operators_per_shift` | Auto-calculated | Override headcount per shift directly |
| `operators_hired` | Auto-calculated | Override total operators hired directly |

In [16]:
# Default fixed OPEX
print(f"\nDefault Fixed OPEX : ${plant.fixed_production_costs:>14,.0f} / yr")

# Override: higher maintenance factor + direct maintenance budget
plant.update_configuration({
    "fixed_opex_factors": {
        "maintenance": 0.07,           # 7 % of ISBL instead of 5 %
        "laboratory_charges": 0.12,    # 12 % of operating labour instead of 10 %
    },
    "fixed_opex_components": {
        "rent_of_land_costs": 100_000,  # fixed annual rent, USD/yr
    },
})

plant.calculate_fixed_opex(print_results=True)
print(plant.fixed_production_costs)


Default Fixed OPEX : $     4,925,445 / yr
Fixed production costs estimation
Operating labor costs: 893,760.00 USD per year
Supervision costs: 223,440.00 USD per year
Direct salary overhead: 558,600.00 USD per year
Laboratory charges: 107,251.20 USD per year
Maintenance costs: 748,667.90 USD per year
Taxes and insurance costs: 160,428.84 USD per year
Rent of land costs: 100,000.00 USD per year
Environmental charges: 143,952.56 USD per year
Operating supplies: 96,257.30 USD per year
General plant overhead: 1,089,270.00 USD per year
Interest on working capital: 233,584.39 USD per year
Patents and royalties: 198,515.56 USD per year
Distribution and selling costs: 198,515.56 USD per year
R&D costs: 297,773.34 USD per year
Fixed OPEX: 5,050,016.64 USD per year
5050016.638051007


In [17]:
# Restore defaults
plant.update_configuration({
    "fixed_opex_factors": {
        "maintenance": None,           
        "laboratory_charges": None,    
    },
    "fixed_opex_components": {
        "rent_of_land_costs": None,  
    },
})
plant.calculate_all()

In [18]:
# --- Labour: auto-calculated baseline ---
default_labor = plant.operating_labor_costs
print(f"Auto-calculated operating labour : ${default_labor:>12,.0f} / yr")
print(f"  Operators per shift            : {plant.calculate_operators_per_shift():.1f}")
print(f"  Operators hired                : {plant.calculate_operators_hired()}")

Auto-calculated operating labour : $     893,760 / yr
  Operators per shift            : 2.6
  Operators hired                : 12


In [19]:
# --- Labour: change shift schedule and hourly rate ---
plant.update_configuration({
    "operator_hourly_rate":      {"rate": 45.0},  # USD/hr
    "working_weeks_per_year":    48,               # fewer working weeks
    "working_shifts_per_week":   5,
    "operating_shifts_per_day":  3,
})

custom_labor = plant.calculate_operating_labor()
print(f"\nCustom shift schedule + rate     : ${custom_labor:>12,.0f} / yr")


Custom shift schedule + rate     : $   1,123,200 / yr


In [20]:
# --- Labour: override headcount directly ---
plant.update_configuration({
    "operators_per_shift": 4,    # fix headcount per shift
    "operators_hired":     18,   # fix total operators hired
})

override_labor = plant.calculate_operating_labor()
print(f"Direct headcount override        : ${override_labor:>12,.0f} / yr")

Direct headcount override        : $   1,555,200 / yr


In [21]:
# Restore defaults
plant.update_configuration({
    "operator_hourly_rate":     {"rate": 38.0},
    "working_weeks_per_year":   49,
    "working_shifts_per_week":  5,
    "operating_shifts_per_day": 3,
    "operators_per_shift":      None,
    "operators_hired":          None,
})
plant.calculate_all()

## Step 8 — Generate the Cash Flow

`calculate_cash_flow()` builds a year-by-year cash flow table covering CAPEX spend, revenue, operating costs, depreciation, taxes, and net cash flow. It returns a `pd.DataFrame` (or `None` if `print_results=True`).

Two optional ramp profiles control the timing:

| Config key | Default | Description |
|---|---|---|
| `capex_ramp` | `[0.3, 0.6, 0.1]` | Fraction of FCI spent in each construction year (must sum to 1) |
| `production_ramp` | `[0, 0, 0.4, 0.8]` | Nameplate capacity fraction for each project year |

In [22]:
# Generate cash flow with default ramp profiles
plant.calculate_cash_flow(print_results=True)

,Year,Capital cost [USD],Revenue [USD],Cash cost [USD],Gross profit [USD],Depreciation [USD],Taxable income [USD],Tax paid [USD],Cash flow [USD]
0,1,"6,261,936.26",0.00,"4,925,445.16","-4,925,445.16",0.00,"-4,925,445.16",0.00,"-11,187,381.41"
1,2,"12,523,872.52",0.00,"4,925,445.16","-4,925,445.16",0.00,"-4,925,445.16",0.00,"-17,449,317.67"
2,3,"5,007,116.91","6,701,400.00","6,875,749.66","-174,349.66","1,652,455.40","-1,826,805.06",0.00,"-5,181,466.57"
3,4,0.00,"13,402,800.00","8,826,054.16","4,576,745.84","1,652,455.40","2,924,290.44",0.00,"4,576,745.84"
4,5,0.00,"16,753,500.00","9,801,206.41","6,952,293.59","1,652,455.40","5,299,838.19","731,072.61","6,221,220.98"
5,6,0.00,"16,753,500.00","9,801,206.41","6,952,293.59","1,652,455.40","5,299,838.19","1,324,959.55","5,627,334.05"
6,7,0.00,"16,753,500.00","9,801,206.41","6,952,293.59","1,652,455.40","5,299,838.19","1,324,959.55","5,627,334.05"
7,8,0.00,"16,753,500.00","9,801,206.41","6,952,293.59","1,652,455.40","5,299,838.19","1,324,959.55","5,627,334.05"
8,9,0.00,"16,753,500.00","9,801,206.41","6,952,293.59","1,652,455.40","5,299,838.19","1,324,959.55","5,627,334.05"
9,10,0.00,"16,753,500.00","9,801,206.41","6,952,293.59","1,652,455.40","5,299,838.19","1,324,959.55","5,627,334.05"


In [23]:
# Custom ramp profiles: 2-year build, gradual production ramp-up
plant.update_configuration({
    "capex_ramp":      [0.4, 0.6],          # 2-year construction
    "production_ramp": [0, 0, 0.5, 0.75],   # full capacity from year 4
})

plant.calculate_cash_flow(print_results=True)

,Year,Capital cost [USD],Revenue [USD],Cash cost [USD],Gross profit [USD],Depreciation [USD],Taxable income [USD],Tax paid [USD],Cash flow [USD]
0,1,"8,349,248.35",0.00,"4,925,445.16","-4,925,445.16",0.00,"-4,925,445.16",0.00,"-13,274,693.50"
1,2,"15,443,677.34",0.00,"4,925,445.16","-4,925,445.16",0.00,"-4,925,445.16",0.00,"-20,369,122.50"
2,3,0.00,"8,376,750.00","7,363,325.78","1,013,424.22","1,652,455.40","-639,031.18",0.00,"1,013,424.22"
3,4,0.00,"12,565,125.00","8,582,266.09","3,982,858.91","1,652,455.40","2,330,403.51",0.00,"3,982,858.91"
4,5,0.00,"16,753,500.00","9,801,206.41","6,952,293.59","1,652,455.40","5,299,838.19","582,600.88","6,369,692.72"
5,6,0.00,"16,753,500.00","9,801,206.41","6,952,293.59","1,652,455.40","5,299,838.19","1,324,959.55","5,627,334.05"
6,7,0.00,"16,753,500.00","9,801,206.41","6,952,293.59","1,652,455.40","5,299,838.19","1,324,959.55","5,627,334.05"
7,8,0.00,"16,753,500.00","9,801,206.41","6,952,293.59","1,652,455.40","5,299,838.19","1,324,959.55","5,627,334.05"
8,9,0.00,"16,753,500.00","9,801,206.41","6,952,293.59","1,652,455.40","5,299,838.19","1,324,959.55","5,627,334.05"
9,10,0.00,"16,753,500.00","9,801,206.41","6,952,293.59","1,652,455.40","5,299,838.19","1,324,959.55","5,627,334.05"


## Step 9 — Change the Depreciation Model

OpenPyTEA supports three depreciation methods, configured via the `depreciation` key:

| Method | Key | Required fields |
|---|---|---|
| Straight-line | `"straight_line"` | `life`, `salvage_fraction`, `service_start_year` |
| Declining balance | `"declining_balance"` | `life`, `salvage_fraction`, `db_factor`, `service_start_year` |
| MACRS | `"macrs"` | `macrs_class` |

Switch between methods via `update_configuration()` — the new config replaces the previous depreciation block entirely.

In [24]:
npv_sl = plant.npv

# --- Declining balance ---
plant.update_configuration({
    "depreciation": {
        "method": "declining_balance",
        "life": 12,
        "salvage_fraction": 0.05,
        "db_factor": 2.0,              # double-declining balance
        "service_start_year": 2,
    }
})

plant.calculate_all()
npv_db = plant.npv

In [25]:
# --- MACRS ---
plant.update_configuration({
    "depreciation": {
        "method": "macrs",
        "macrs_class": 7,              # 7-year MACRS property
    }
})

plant.calculate_all()
npv_macrs = plant.npv

In [28]:
print("NPV by depreciation method:")
print(f"  Straight-line     : ${npv_sl:>14,.0f}")
print(f"  Declining balance : ${npv_db:>14,.0f}")
print(f"  MACRS (7-yr)      : ${npv_macrs:>14,.0f}")

NPV by depreciation method:
  Straight-line     : $    11,054,372
  Declining balance : $    11,179,209
  MACRS (7-yr)      : $    11,466,265


## Summary

Building and configuring a `Plant` involves these steps:

1. **Create `Equipment` objects** (Tutorial 1)
2. **Write a configuration dictionary** — economic parameters, products, operating costs
3. **Instantiate `Plant(config)`** and call `calculate_all()`
4. **Inspect results** — FCI, OPEX, revenue, and financial metrics
5. **Update configuration** — change parameters without rebuilding the plant
6. **Override capital cost factors** — `fixed_capital_factors` (multipliers), `fixed_capital_components` (direct values), `loc_factor`
7. **Override fixed OPEX and labour** — via `fixed_opex_factors`, `fixed_opex_components`, shift schedule, and headcount
8. **Generate cash flow** — with `capex_ramp` and `production_ramp` profiles
9. **Switch depreciation model** — straight-line, declining balance, or MACRS

Key financial outputs:

| Method | Output |
|---|---|
| `calculate_fixed_capital()` | Fixed capital investment (FCI) |
| `calculate_variable_opex()` | Annual variable operating cost |
| `calculate_fixed_opex()` | Annual fixed operating cost |
| `calculate_revenue()` | Annual revenue from plant products |
| `calculate_cash_flow()` | Year-by-year cash flow DataFrame |
| `calculate_levelized_cost()` | LCOP (USD per unit of product) |
| `calculate_npv()` | Net present value |
| `calculate_irr()` | Internal rate of return |
| `calculate_roi()` | Return on investment |
| `calculate_payback_time()` | Payback period (years) |

**Next:** Tutorial 3 — Performing Analysis shows how to run sensitivity analysis, tornado diagrams, and Monte Carlo simulations.